# Imagenes de test del modelo de clasificacion

Este notebook saca imagenes del **test** de `modelo_clasificacion2`, el mismo test que se uso en el entrenamiento (`cnn_damage_classification_multiclass_training.ipynb`). Son imagenes que el modelo nunca vio al entrenar.

El split se rearma con el mismo csv y los mismos `train_test_split` con `random_state=42`, asi cada imagen cae en el mismo split que al entrenar. Despues se eligen 5 imagenes por superclase y se copian a `imagenes_test_clasificacion/`, una carpeta por clase, con un csv que dice de donde salio cada una.

Antes hay que tener el dataset descargado con `cnn_damage_classification_build_dataset.ipynb` (`data_clasificacion/dataset_final.csv` y las imagenes). Correr las celdas en orden.

In [ ]:
# Celda 1: imports y configuracion
# N_POR_CLASE es cuantas imagenes se sacan de cada superclase.
# SEED_MUESTRA fija cuales se eligen dentro del test: con el mismo numero
# siempre salen las mismas, cambiarlo para tener otras (siguen siendo de test).
# SALIDA es la carpeta donde se copian las imagenes.

import os
import json
import shutil

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

DATASET_CSV = "data_clasificacion/dataset_final.csv"
CLASES_JSON = "../models/modelo_clasificacion2_clases.json"
GRUPOS_JSON = "../models/modelo_clasificacion2_grupos.json"

N_POR_CLASE = 5
SEED_MUESTRA = 42
SALIDA = "imagenes_test_clasificacion"

with open(CLASES_JSON, encoding="utf-8") as f:
    FINAL_CLASSES = json.load(f)
print(f"Clases del modelo: {len(FINAL_CLASSES)}")

In [ ]:
# Celda 2: verificar el dataset
# El split solo es el mismo del entrenamiento si el csv es el mismo, por
# eso se chequea:
#   - que exista dataset_final.csv con las columnas image_path y label
#   - que tenga las mismas 11 clases del modelo
#   - que grupos_final.json sea igual a modelo_clasificacion2_grupos.json
#   - que las imagenes esten en disco
# Si algo falla corta con un error que dice que falta.
# Depende de: DATASET_CSV, GRUPOS_JSON, FINAL_CLASSES (celda 1)

if not os.path.exists(DATASET_CSV):
    raise FileNotFoundError(f"No existe {DATASET_CSV}: correr antes cnn_damage_classification_build_dataset.ipynb")

df = pd.read_csv(DATASET_CSV)
problemas = []

faltan_columnas = {"image_path", "label"} - set(df.columns)
if faltan_columnas:
    raise ValueError(f"Al csv le faltan las columnas {sorted(faltan_columnas)}")

if set(df["label"]) != set(FINAL_CLASSES):
    problemas.append(f"las clases del csv no son las del modelo: {sorted(set(df['label']) ^ set(FINAL_CLASSES))}")

grupos_path = os.path.join(os.path.dirname(DATASET_CSV), "grupos_final.json")
with open(GRUPOS_JSON, encoding="utf-8") as f:
    grupos_modelo = json.load(f)
if not os.path.exists(grupos_path):
    problemas.append(f"no existe {grupos_path}")
else:
    with open(grupos_path, encoding="utf-8") as f:
        if json.load(f) != grupos_modelo:
            problemas.append("grupos_final.json no coincide con los grupos del modelo (el dataset es de otro ETL)")

no_estan = [r for r in df["image_path"] if not os.path.exists(r)]
if no_estan:
    problemas.append(f"{len(no_estan)} de {len(df)} imagenes no estan en disco (ej: {no_estan[0]})")

if problemas:
    raise ValueError("El dataset no sirve para este modelo:\n  - " + "\n  - ".join(problemas))

print(f"Dataset OK: {len(df)} imagenes, todas en disco, mismas clases y grupos que el modelo")

In [ ]:
# Celda 3: rearmar el split train/val/test (70/15/15)
# Exactamente igual que la celda 2 de cnn_damage_classification_multiclass_training:
# primero 70% train / 30% resto, y el resto a la mitad entre val y test,
# estratificado por label y con random_state=42. Con el mismo csv y la
# misma semilla la mezcla sale igual, asi que test_df es el test real.
# No cambiar nada de esta celda.
# Depende de: df (celda 2)

train_df, resto_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(resto_df, test_size=0.50, stratify=resto_df["label"], random_state=42)

print(f"train: {len(train_df)} | val: {len(val_df)} | test: {len(test_df)}\n")
print(test_df["label"].value_counts().reindex(FINAL_CLASSES).to_string())

In [ ]:
# Celda 4: elegir N_POR_CLASE imagenes por clase y copiarlas
# Toma al azar (con SEED_MUESTRA) N_POR_CLASE imagenes de cada clase del
# test y las copia a SALIDA/<clase>/. Como algunos nombres de clase tienen
# "/" (ej: "Material loss/erosion"), en el nombre de la carpeta se cambia
# por "_". Las imagenes originales no se tocan, solo se copian.
# Si SALIDA ya existe se borra antes, para que no queden imagenes de una
# corrida anterior mezcladas.
# Al final guarda SALIDA/imagenes_test.csv con la clase real, la ruta
# original y la ruta de la copia de cada imagen.
# Depende de: test_df (celda 3), N_POR_CLASE, SEED_MUESTRA, SALIDA (celda 1)

muestra = (test_df.groupby("label", group_keys=False)
           .sample(n=N_POR_CLASE, random_state=SEED_MUESTRA)
           .set_index("label").loc[FINAL_CLASSES].reset_index())

if os.path.exists(SALIDA):
    shutil.rmtree(SALIDA)

copias = []
for _, fila in muestra.iterrows():
    carpeta = os.path.join(SALIDA, fila["label"].replace("/", "_"))
    os.makedirs(carpeta, exist_ok=True)
    destino = os.path.join(carpeta, os.path.basename(fila["image_path"])).replace(os.sep, "/")
    shutil.copy2(fila["image_path"], destino)
    copias.append(destino)

muestra["copia"] = copias
muestra[["label", "image_path", "copia"]].to_csv(f"{SALIDA}/imagenes_test.csv", index=False)

print(f"Se copiaron {len(muestra)} imagenes ({N_POR_CLASE} por clase) a '{SALIDA}/'")
print(f"Lista en {SALIDA}/imagenes_test.csv")

In [ ]:
# Celda 5: ver las imagenes elegidas
# Una fila por clase con sus N_POR_CLASE imagenes, para chequear que se
# copio lo que se esperaba.
# Depende de: muestra (celda 4), FINAL_CLASSES, N_POR_CLASE (celda 1)

fig, axes = plt.subplots(len(FINAL_CLASSES), N_POR_CLASE,
                         figsize=(2.6 * N_POR_CLASE, 2.6 * len(FINAL_CLASSES)))
for i, clase in enumerate(FINAL_CLASSES):
    rutas = muestra[muestra["label"] == clase]["copia"].tolist()
    for j, ax in enumerate(axes[i]):
        ax.axis("off")
        if j < len(rutas):
            ax.imshow(Image.open(rutas[j]).convert("RGB"))
        if j == 0:
            ax.set_title(clase, fontsize=9, loc="left")
plt.tight_layout()
plt.show()